# Week 13: Reinforcement Learning & RLHF
## Notebook 07: Kahneman-Tversky Optimization (KTO) vs DPO Alignment

### 1. Conceptual Intuition & Theoretical Background
While Direct Preference Optimization (DPO) simplifies RLHF by eliminating separate reward models, it requires **pairwise preference datasets** $(x, y_w, y_l)$ where an explicit chosen output $y_w$ is paired with a rejected output $y_l$ for the same prompt.

In real-world production, user feedback is **unpaired and binary**: a user clicks "thumbs up" or "thumbs down" on an isolated model response.

**Kahneman-Tversky Optimization (KTO)** aligns models directly on unpaired binary feedback by leveraging human **Prospect Theory**:
Humans evaluate outcomes relative to a reference point $z_0$ and are **loss averse** (losses hurt more than equal gains feel good).

Implicit reward under reference policy $\pi_{\text{ref}}$:
$$r_\theta(x, y) = \beta \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)}$$

KTO Loss:
$$\mathcal{L}_{\text{KTO}} = \mathbb{E}_{x, y} \left[ w(y) \cdot \left( 1 - v_\theta(x, y) \right) \right]$$
Where value function $v_\theta$ incorporates Kahneman-Tversky asymmetric scaling:
$$v(z) = \begin{cases} \sigma(\beta(z - z_0)) & \text{if desirable (thumbs up)} \\ \sigma(\beta(z_0 - z)) & \text{if undesirable (thumbs down)} \end{cases}$$

---

### 2. Scratch Python Implementation: KTO vs DPO Objective

In [ ]:
import numpy as np

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -25, 25)))

# 1. DPO Loss (Pairwise Chosen vs Rejected)
def dpo_loss(pi_chosen_logp, pi_rejected_logp, ref_chosen_logp, ref_rejected_logp, beta=0.1):
    log_ratio_chosen = pi_chosen_logp - ref_chosen_logp
    log_ratio_rejected = pi_rejected_logp - ref_rejected_logp
    
    logits = beta * (log_ratio_chosen - log_ratio_rejected)
    loss = -np.mean(np.log(sigmoid(logits) + 1e-9))
    return loss

# 2. KTO Loss (Unpaired Binary Thumbs Up / Down)
def kto_loss(log_ratios, labels, z0=0.0, beta=0.1, lambda_D=1.0, lambda_U=1.33):
    """
    log_ratios: log(pi_theta / pi_ref)
    labels: +1 for thumbs up (desirable), -1 for thumbs down (undesirable)
    lambda_U > lambda_D captures human loss aversion!
    """
    losses = []
    for r, label in zip(log_ratios, labels):
        if label == 1:
            # Desirable: v = sigmoid(beta * (r - z0))
            v = sigmoid(beta * (r - z0))
            loss = lambda_D * (1.0 - v)
        else:
            # Undesirable: v = sigmoid(beta * (z0 - r))
            v = sigmoid(beta * (z0 - r))
            loss = lambda_U * (1.0 - v)
        losses.append(loss)
    return np.mean(losses)

# Benchmark comparison on synthetic policy ratios
np.random.seed(42)
n_samples = 100

# Simulated log probabilities
pi_good = np.random.normal(loc=-1.2, scale=0.3, size=n_samples)
ref_good = np.random.normal(loc=-1.5, scale=0.3, size=n_samples)

pi_bad = np.random.normal(loc=-2.0, scale=0.3, size=n_samples)
ref_bad = np.random.normal(loc=-1.8, scale=0.3, size=n_samples)

loss_dpo = dpo_loss(pi_good, pi_bad, ref_good, ref_bad, beta=0.1)

# Format for KTO: Unpaired single responses with +1 or -1
ratios = np.concatenate([pi_good - ref_good, pi_bad - ref_bad])
binary_labels = np.array([1]*n_samples + [-1]*n_samples)

loss_kto = kto_loss(ratios, binary_labels, z0=0.0, beta=0.1, lambda_D=1.0, lambda_U=1.33)

print(f"DPO Pairwise Loss:           {loss_dpo:.4f}")
print(f"KTO Unpaired Loss:           {loss_kto:.4f}")
print("KTO successfully evaluates alignment using isolated binary labels without pair construction!")

### 3. Step-by-Step Code Walkthrough
1. **Unpaired Input Structure**: Eliminates the need to construct balanced prompt-matched pairs.
2. **Loss Aversion Weighting**: $\lambda_U = 1.33$ penalizes undesirable outputs more heavily than it rewards desirable outputs.
3. **Reference Anchor ($z_0$)**: Calibrates the neutral boundary score.

---

### 4. Industry Applications & Trade-offs
- **Real-World Product Alignment:** Directly ingesting production user thumbs-up/down click streams without pairing data.
- **Trade-off:** Sensitive to the baseline offset hyperparameter $z_0$.

---

### 5. Top Interview Questions & Answers
**Q1: What practical operational bottleneck does KTO eliminate compared to DPO?**
> **Answer:** DPO requires pairwise data where two different outputs are generated for the exact same prompt and ranked. KTO trains on individual responses labeled with binary feedback (thumbs-up/thumbs-down), matching real-world user interaction telemetry.

**Q2: How does KTO incorporate behavioral economics (Prospect Theory)?**
> **Answer:** It implements human loss aversion by setting the weight for undesirable outputs ($\lambda_U$) higher than desirable outputs ($\lambda_D$), reflecting that human dissatisfaction with a bad output outweighs satisfaction with a good one.